In [201]:
import numpy as np
import pandas as pd
import os
from sklearn.preprocessing import MinMaxScaler
from utils.paths import DATASETS_CLEANED_DIR, DATASETS_PREPROCESSED_DIR

In [202]:
datasets = []

for file in os.listdir(DATASETS_CLEANED_DIR):
    if file.endswith(".csv"):
        df = pd.read_csv(os.path.join(DATASETS_CLEANED_DIR, file))
        datasets.append(df)

In [203]:
# converting degrees to sin and cos for wind direction
heights = [10, 100]

for i, df in enumerate(datasets):
    for h in heights:
        col_name = f"wind_direction_{h}m"

        radians = np.radians(df[col_name])
        df[f"wind_sin_{h}m"] = np.sin(radians)
        df[f"wind_cos_{h}m"] = np.cos(radians)

        df.drop(columns=[col_name], inplace=True)

    datasets[i] = df

In [204]:
# converting from Fahrenheit to Celsius
for i, df in enumerate(datasets):
    df['temperature_2m'] = (df['temperature_2m'] - 32) * 5/9

    datasets[i] = df

In [205]:
# finding v_cutin and v_rated values

v_cutin_list = []
v_rated_list = []

n_percentile = 1

for i, df in enumerate(datasets):
    working_turbines_map = df[df['power'] > 0]

    v_cutin = np.percentile(working_turbines_map['wind_speed_100m'], n_percentile)
    v_rated = np.percentile(working_turbines_map['wind_speed_100m'], 100 - n_percentile)

    v_cutin_list.append(v_cutin)
    v_rated_list.append(v_rated)

    print(f"Dataset {i+1}: v_cutin = {v_cutin:.2f} m/s, v_rated = {v_rated:.2f} m/s")

Dataset 1: v_cutin = 3.10 m/s, v_rated = 13.52 m/s
Dataset 2: v_cutin = 3.14 m/s, v_rated = 16.79 m/s
Dataset 3: v_cutin = 3.12 m/s, v_rated = 14.82 m/s
Dataset 4: v_cutin = 3.11 m/s, v_rated = 14.03 m/s


In [206]:
# normalizing wind_speed columns

heights = [10, 100]
scaler = MinMaxScaler()

for i, df in enumerate(datasets):

    scaler.data_min_ = np.array([v_cutin_list[i]])
    scaler.data_max_ = np.array([v_rated_list[i]])
    scaler.data_range_ = scaler.data_max_ - scaler.data_min_
    scaler.scale_ = (1.0 - 0.0) / scaler.data_range_
    scaler.min_ = 0.0 - scaler.data_min_ * scaler.scale_

    for h in heights:
        v_norm = scaler.transform([df[f'wind_speed_{h}m']])
        v_norm = np.clip(v_norm, 0, 1)  # Ensure values are within [0, 1]
        
        df[f'wind_speed_{h}m'] = np.transpose(v_norm)

    datasets[i] = df

In [207]:
# correcting still invalid records, where wind speed is 0 but power is greater than 0
for i, df in enumerate(datasets):
    df.loc[df['wind_speed_100m'] == 0, 'power'] = 0
    datasets[i] = df

In [208]:
for i, df in enumerate(datasets):
    print(f"Location_{i+1}:")
    print(df[(df['wind_speed_100m'] == 0) & (df['power'] > 0)]['power'])
    print(df[(df['wind_speed_100m'] == 0) & (df['power'] > 0)]['power'].max())
    print(f"\n{'#' * 40}\n")

Location_1:
Series([], Name: power, dtype: float64)
nan

########################################

Location_2:
Series([], Name: power, dtype: float64)
nan

########################################

Location_3:
Series([], Name: power, dtype: float64)
nan

########################################

Location_4:
Series([], Name: power, dtype: float64)
nan

########################################



In [209]:
# splitting into train, validation and test sets

train_sets = []
val_sets = []
test_sets = []
split_ratios = [0.6, 0.2, 0.2]  # 3 years for training, 1 year for val and 1 year for test

for df in datasets:
    n = len(df)
    train_end = int(split_ratios[0] * n)
    val_end = train_end + int(split_ratios[1] * n)

    train_df = df.iloc[:train_end]
    val_df = df.iloc[train_end:val_end]
    test_df = df.iloc[val_end:]

    train_sets.append(train_df)
    val_sets.append(val_df)
    test_sets.append(test_df)

In [210]:
# scaling columns

cols_to_scale = ['temperature_2m', 'dew_point_2m', 'wind_gusts_10m', 'relative_humidity_2m']

for i in range(len(train_sets)):
    scaler = MinMaxScaler()

    train_sets[i][cols_to_scale] = scaler.fit_transform(train_sets[i][cols_to_scale])
    val_sets[i][cols_to_scale] = scaler.transform(val_sets[i][cols_to_scale])
    test_sets[i][cols_to_scale] = scaler.transform(test_sets[i][cols_to_scale])

In [211]:
# calculating the empirical curve

def apply_empirical_curve(df, curve):
    df['v_bin'] = df['wind_speed_100m'].round(2)
    df['p_base'] = df['v_bin'].map(curve)

    return df

for i in range(len(train_sets)):
    train_sets[i]['v_bin'] = train_sets[i]['wind_speed_100m'].round(2)
    empirical_curve = train_sets[i].groupby('v_bin')['power'].mean()
    
    train_sets[i] = apply_empirical_curve(train_sets[i], empirical_curve)
    val_sets[i] = apply_empirical_curve(val_sets[i], empirical_curve)
    test_sets[i] = apply_empirical_curve(test_sets[i], empirical_curve)

In [212]:
# calculating the delta_p

for i in range(len(train_sets)):
    train_sets[i]['delta_p'] = train_sets[i]['power'] - train_sets[i]['p_base']
    val_sets[i]['delta_p'] = val_sets[i]['power'] - val_sets[i]['p_base']
    test_sets[i]['delta_p'] = test_sets[i]['power'] - test_sets[i]['p_base']

In [213]:
# saving the preprocessed datasets

for df in train_sets + val_sets + test_sets:
    df.drop(columns=['v_bin', 'power',], inplace=True)

os.makedirs(DATASETS_PREPROCESSED_DIR, exist_ok=True)

for i in range(len(train_sets)):
    train_sets[i].to_csv(os.path.join(DATASETS_PREPROCESSED_DIR, f"Location_{i+1}_train.csv"), index=False)
    val_sets[i].to_csv(os.path.join(DATASETS_PREPROCESSED_DIR, f"Location_{i+1}_val.csv"), index=False)
    test_sets[i].to_csv(os.path.join(DATASETS_PREPROCESSED_DIR, f"Location_{i+1}_test.csv"), index=False)

In [214]:
# sanity check

for file in os.listdir(DATASETS_PREPROCESSED_DIR):
    if file.endswith(".csv"):
        df = pd.read_csv(os.path.join(DATASETS_PREPROCESSED_DIR, file))
        
        print(f"{file}:")
        print(df.describe())
        print(df.info())
        print(df.isna().sum())
        print("#" * 40)

Location_1_train.csv:
       temperature_2m  relative_humidity_2m  dew_point_2m  wind_speed_10m  \
count    26280.000000          26280.000000  26280.000000    26280.000000   
mean         0.566199              0.666893      0.589587        0.087830   
std          0.182106              0.203848      0.207625        0.123360   
min          0.000000              0.000000      0.000000        0.000000   
25%          0.422120              0.512195      0.431478        0.000000   
50%          0.561290              0.695122      0.589936        0.021113   
75%          0.720737              0.841463      0.765525        0.144914   
max          1.000000              1.000000      1.000000        0.993282   

       wind_speed_100m  wind_gusts_10m  wind_sin_10m  wind_cos_10m  \
count     26280.000000    26280.000000  26280.000000  26280.000000   
mean          0.318641        0.261916     -0.234107     -0.052926   
std           0.238894        0.128516      0.695580      0.677198   
min 